# 11 · (Optional) Monitoring conditions: degraded footage + your own CCTV images

**Purpose.** Proxy for surveillance / traffic footage: apply low-light, motion-blur, low-resolution and JPEG degradations to a test subset (COCO labels stay valid) and compare detector-only vs the pipeline using the thresholds selected on clean held-out data. A second section runs the online pipeline on a folder of your own images (qualitative).

| | |
|---|---|
| **Proposal objective** | Optional extension (proposal §6.1) |
| **Inputs** | COCO test subset, optional image folder |
| **Outputs** | `results/robustness/*.csv` |
| **Runtime (T4)** | ≈15–25 min per condition (200 images) |

> Run cells top to bottom. Every expensive stage is **resumable**: if Colab disconnects, reconnect,
> re-run from the top and it continues where it stopped (results are cached on Drive under `results/`).

In [ ]:
# ===================== Bootstrap (identical in every notebook) =====================
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules

# Where the project lives on Google Drive. Upload the whole repo folder there (see docs/COLAB_GUIDE.md),
# or set REPO_URL and it will be cloned into DRIVE_PROJECT on first run.
DRIVE_PROJECT = "/content/drive/MyDrive/detect-verify-decide"
REPO_URL = ""  # e.g. "https://github.com/<you>/detect-verify-decide.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT) and REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, DRIVE_PROJECT], check=True)
    PROJECT_ROOT = DRIVE_PROJECT
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src", "dvd")), f"src/dvd not found under {PROJECT_ROOT}"
os.environ["DVD_ROOT"] = PROJECT_ROOT
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))
print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
# Install dependencies (Colab already has torch/torchvision with CUDA). ~1-2 min per fresh runtime.
if IN_COLAB:
    %pip install -q -r "{PROJECT_ROOT}/requirements.txt"

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from dvd.config import load_config, save_selected, results_dir
from dvd.io_utils import set_seed, save_json, load_json, JsonlCache
from dvd import paths

cfg = load_config()
set_seed(cfg["seed"])
pd.set_option("display.precision", 4)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only - expect very slow runs)")
except ImportError:
    print("torch not installed")
print("data_root   :", cfg["paths"]["data_root"])
print("results_root:", cfg["paths"]["results_root"])
print("tuned params:", cfg["selected"] or "(none yet)")

In [ ]:
from dvd.data.coco import prepare_coco, CocoData, load_splits
prepare_coco(cfg)          # after a runtime restart this re-extracts from the zip cached on Drive (~1-2 min)
coco = CocoData(cfg)
splits = load_splits(cfg)
HELDOUT, TEST = splits["heldout"], splits["test"]
print(f"held-out: {len(HELDOUT)} images | test: {len(TEST)} images")

In [ ]:
from dvd.degrade import degrade
from dvd.agents.detector import build_detector
from dvd.agents.verifier import build_verifier
from dvd.pipeline import run_detection_stage, run_verification_stage
from dvd.experiments import SplitData, evaluate_row, cached
from dvd.metrics import decision_breakdown
SUB = TEST[: cfg["robustness"]["images"]]
detector = build_detector(cfg, coco.cat_names, coco.name_to_cat_id); verifier = build_verifier(cfg)
iid = SUB[0]
fig, ax = plt.subplots(1, 5, figsize=(22, 4))
for a, c in zip(ax, ["clean"] + cfg["robustness"]["conditions"]):
    a.imshow(coco.image(iid) if c == "clean" else degrade(coco.image(iid), c, iid)); a.set_title(c); a.axis("off")
plt.show()

In [ ]:
sel = cfg["selected"]; rows = []
for cond in ["clean"] + cfg["robustness"]["conditions"]:
    # clean images reuse the test-split caches from notebooks 02/03 (already computed -> no-op)
    split, tag = ("test", "") if cond == "clean" else ("robust", f"_{cond}")
    fn = coco.image if cond == "clean" else (lambda i, c=cond: degrade(coco.image(i), c, i))
    det = run_detection_stage(detector, SUB, paths.det_cache(cfg, split, tag), fn, f"detect {cond}")
    run_verification_stage(verifier, det, SUB, paths.ver_cache(cfg, split, "crop", tag=tag), cfg, fn, view="crop", desc=f"verify {cond}")
    D = SplitData(cfg, coco, split, SUB, det_tag=tag)
    v = cached(paths.ver_cache(cfg, split, "crop", tag=tag), SUB)
    meth = {"kind": "rule", "verdicts": {i: np.asarray(r["verdict"], bool) for i, r in v.items()}}
    base = evaluate_row({"kind": "none"}, D, {"tau": sel["detector_only_tau"]})
    dvd = evaluate_row(meth, D, {"tau_low": sel["tau_low"], "tau_high": sel["tau_high"]})
    b = decision_breakdown(D.labels, {i: D.ev.scores(i) for i in SUB}, dvd.pop("_masks"), sel["tau_low"])
    base.pop("_masks")
    rows.append({"condition": cond, "det_precision": base["precision"], "det_recall": base["recall"], "det_fp_per_img": base["fp_per_image"],
                 "dvd_precision": dvd["precision"], "dvd_recall": dvd["recall"], "dvd_fp_per_img": dvd["fp_per_image"],
                 "fp_reduction": 1 - dvd["fp"] / max(base["fp"], 1), "over_rejection_rate": b["over_rejection_rate"]})
rob = pd.DataFrame(rows); rob.to_csv(results_dir(cfg, "robustness") / "degradations.csv", index=False); rob

## Your own monitoring images (qualitative)
Put `.jpg/.png` frames in `data/custom_images/` on Drive (no labels needed).

In [ ]:
from pathlib import Path
from PIL import Image
from dvd.agents.decision import RuleDecisionAgent
from dvd.pipeline import DetectVerifyDecide
from dvd.viz import show_detections
CUSTOM_DIR = Path(cfg["root"]) / "data" / "custom_images"
files = sorted([p for p in CUSTOM_DIR.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:8] if CUSTOM_DIR.exists() else []
print(f"{len(files)} custom images found in {CUSTOM_DIR}")
pipe = DetectVerifyDecide(detector, verifier, RuleDecisionAgent(sel["tau_low"], sel["tau_high"]), cfg)
for p in files:
    img = Image.open(p).convert("RGB"); r = pipe(img)
    keep = [k for k, s in enumerate(r["scores"]) if s >= sel["tau_low"]]
    show_detections(img, [r["boxes"][k] for k in keep], [r["labels"][k] for k in keep], [r["scores"][k] for k in keep],
                    ["accepted" if r["accept"][k] else "rejected" for k in keep], title=f"{p.name} - {r['total_time_s']:.2f}s"); plt.show()